## Filter out records on land (20 km buffer)

In [ ]:
# Enable tqdm for pandas
tqdm.pandas()
# Step 1: Load the Natural Earth 10m land polygons
print("Loading Natural Earth 10m land polygons...")
land_polygons = gpd.read_file("NE_10m_land/NE_10m_land.shp")
print("Land polygons loaded.")
# Step 2: Apply a 20 km inward buffer to the land polygons
print("Applying 20 km inward buffer to land polygons...")
land_polygons = land_polygons.to_crs("EPSG:3395")  # Project to a metric CRS (e.g., Mercator)
land_polygons['geometry'] = land_polygons.geometry.buffer(-20000)  # Negative buffer (20 km inward)
land_polygons = land_polygons.to_crs("EPSG:4326")  # Reproject back to WGS84
print("Buffering complete.")
# Step 3: Load the GBIF dataset using dask
print("Processing GBIF dataset in chunks...")
gbif_file = "gbif_combined_dataset.parquet"
output_file = "gbif_filtered_dataset_marine.parquet"
# Load the Parquet file as a Dask DataFrame
ddf = dd.read_parquet(gbif_file)
# Define a function to process each partition
def process_partition(partition):
    if 'decimalLatitude' not in partition.columns or 'decimalLongitude' not in partition.columns:
        raise ValueError("The dataset must contain 'decimalLatitude' and 'decimalLongitude' columns.")
    
    # Create geometry column from latitude and longitude
    geometry = [Point(xy) for xy in zip(partition['decimalLongitude'], partition['decimalLatitude'])]
    partition_gdf = gpd.GeoDataFrame(partition, geometry=geometry, crs="EPSG:4326")
    
    # Perform spatial filtering using spatial indexing
    # Exclude records that are within the buffered land polygons
    partition_gdf = partition_gdf.sjoin(land_polygons, how="left", predicate="within")
    partition_gdf = partition_gdf[partition_gdf['index_right'].isna()]  # Keep only records NOT within land polygons
    
    # Drop unnecessary columns (e.g., geometry and columns from land_polygons)
    partition_gdf.drop(columns=['geometry', 'featurecla', 'index_right', 'min_zoom', 'scalerank'], inplace=True, errors='ignore')
    
    return partition_gdf
# Define metadata for the output DataFrame
meta = ddf.head(0)  # Start with the original metadata
meta = meta.drop(columns=['geometry'], errors='ignore')  # Drop geometry column if it exists
# Apply the processing function to each partition
filtered_ddf = ddf.map_partitions(process_partition, meta=meta)
# Compute the result and save it to a Parquet file
filtered_ddf.compute().to_parquet(output_file)
print(f"Filtered dataset saved as '{output_file}'")

## Column names

In [ ]:
# Replace 'your_file.txt' with the path to your tab-delimited file
input_file_path = 'filtered_data_eurasia.txt'
output_file_path = 'GBIF_columns_eurasia.csv'
# Open the file and read the first line to get the column names
with open(input_file_path, 'r') as file:
    first_line = file.readline()
# Split the first line based on the tab delimiter
columns = first_line.strip().split('\t')
# Save the column names to a CSV file
with open(output_file_path, 'w', newline='') as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(['Column Names'])  # Add a header for the CSV
    for column in columns:
        writer.writerow([column])
print(f"Column names have been saved to {output_file_path}")